<a href="https://colab.research.google.com/github/Raksh1707/taskdeeplearning/blob/main/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from scipy import signal


In [2]:
def im2col(input_data, filter_h, filter_w, stride=1, padding=0):

    N, C, H, W = input_data.shape

    out_h = (H + 2 * padding - filter_h) // stride + 1
    out_w = (W + 2 * padding - filter_w) // stride + 1

    img = np.pad(
        input_data,
        [(0,0), (0,0), (padding,padding), (padding,padding)],
        mode='constant'
    )

    col = np.zeros((N, C, filter_h, filter_w, out_h, out_w))

    for y in range(filter_h):
        y_max = y + stride * out_h
        for x in range(filter_w):
            x_max = x + stride * out_w
            col[:, :, y, x, :, :] = img[:, :, y:y_max:stride, x:x_max:stride]

    col = col.transpose(0,4,5,1,2,3).reshape(N*out_h*out_w, -1)

    return col

In [3]:
# Convolution Layer
# -----------------------------
class Conv2D:

    def __init__(self, in_channels, out_channels,
                 kernel_size, stride=1, padding=0):

        self.in_channels = in_channels
        self.out_channels = out_channels

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

        self.weights = np.random.randn(
            out_channels,
            in_channels,
            kernel_size,
            kernel_size
        ) * 0.01

        self.bias = np.zeros(out_channels)

    def forward(self, x):

        N, C, H, W = x.shape

        FH = self.kernel_size
        FW = self.kernel_size

        out_h = (H + 2*self.padding - FH)//self.stride + 1
        out_w = (W + 2*self.padding - FW)//self.stride + 1

        col = im2col(
            x,
            FH,
            FW,
            self.stride,
            self.padding
        )

        col_w = self.weights.reshape(self.out_channels, -1).T

        out = np.dot(col, col_w) + self.bias

        out = out.reshape(N, out_h, out_w, self.out_channels)

        out = out.transpose(0,3,1,2)

        return out

In [4]:
np.random.seed(42)

# Batch=2
# Channels=3
# Height=32
# Width=32
input_tensor = np.random.randn(2,3,32,32)

conv = Conv2D(
    in_channels=3,
    out_channels=8,
    kernel_size=3,
    stride=1,
    padding=1
)

output = conv.forward(input_tensor)

print("Input Shape :", input_tensor.shape)
print("Output Shape:", output.shape)

print("\nOutput Sample:")
print(output[0,0,:5,:5])

Input Shape : (2, 3, 32, 32)
Output Shape: (2, 8, 32, 32)

Output Sample:
[[-0.01394818  0.02260502 -0.02195407 -0.02042987 -0.02689788]
 [ 0.00793353  0.01940568 -0.03704846 -0.00849125  0.05274211]
 [ 0.00275803 -0.03769601 -0.04657203  0.04135658  0.02674725]
 [-0.05584152  0.05883527  0.01045395 -0.04601327  0.02984025]
 [ 0.0310706  -0.03026632  0.00020672  0.04016833  0.00142989]]
